In [5]:

from pathlib import Path
import duckdb

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
CSV = DATA_DIR / "olist_orders_dataset.csv"

print(duckdb.sql(f"SELECT count(*) FROM '{CSV}'"))


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



In [6]:

from pathlib import Path

DATA_DIR = Path.cwd() / "data" / "raw"

for file in sorted(DATA_DIR.glob("*.csv")):
    print(file.name)


olist_customers_dataset.csv
olist_order_items_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv


In [7]:

import duckdb
from pathlib import Path

DATA_DIR = Path.cwd() / "data" / "raw"

tables = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv"
}

for table, filename in tables.items():
    csv_path = (DATA_DIR / filename).as_posix()

    print(f"\n{'=' * 50}")
    print(f"TABLE: {table}")
    print("=" * 50)

    print("\nColumn structure:")
    print(duckdb.sql(
        f"DESCRIBE SELECT * FROM read_csv_auto('{csv_path}')"
    ))

    print("\nFirst 5 rows:")
    print(duckdb.sql(
        f"SELECT * FROM read_csv_auto('{csv_path}') LIMIT 5"
    ))



TABLE: customers

Column structure:
┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_unique_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_zip_code_prefix │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_city            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_state           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘


First 5 rows:
┌──────────────────────────────────┬──────────────────────────────────┬──────────────────────────┬───────────────────

In [16]:
query = """
SELECT 'customers' AS tbl, COUNT(*) AS total_rows, COUNT(customer_id) AS non_null_ids, COUNT(DISTINCT customer_id) AS distinct_ids FROM raw.customers
UNION ALL
SELECT 'orders', COUNT(*), COUNT(order_id), COUNT(DISTINCT order_id) FROM raw.orders
UNION ALL
SELECT 'products', COUNT(*), COUNT(product_id), COUNT(DISTINCT product_id) FROM raw.products
UNION ALL
SELECT 'sellers', COUNT(*), COUNT(seller_id), COUNT(DISTINCT seller_id) FROM raw.sellers
"""

print(duckdb.sql(query))

┌───────────┬────────────┬──────────────┬──────────────┐
│    tbl    │ total_rows │ non_null_ids │ distinct_ids │
│  varchar  │   int64    │    int64     │    int64     │
├───────────┼────────────┼──────────────┼──────────────┤
│ customers │      99441 │        99441 │        99441 │
│ orders    │      99441 │        99441 │        99441 │
│ products  │      32951 │        32951 │        32951 │
│ sellers   │       3095 │         3095 │         3095 │
└───────────┴────────────┴──────────────┴──────────────┘



In [17]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(order_id) AS non_null_order_id,
    COUNT(order_item_id) AS non_null_order_item_id,
    COUNT(DISTINCT order_id) AS distinct_order_id,
    COUNT(DISTINCT (order_id, order_item_id)) AS distinct_pairs
FROM raw.order_items
"""

print(duckdb.sql(query))

┌────────────┬───────────────────┬────────────────────────┬───────────────────┬────────────────┐
│ total_rows │ non_null_order_id │ non_null_order_item_id │ distinct_order_id │ distinct_pairs │
│   int64    │       int64       │         int64          │       int64       │     int64      │
├────────────┼───────────────────┼────────────────────────┼───────────────────┼────────────────┤
│     112650 │            112650 │                 112650 │             98666 │         112650 │
└────────────┴───────────────────┴────────────────────────┴───────────────────┴────────────────┘



In [18]:
for t in ["customers", "orders", "order_items", "products", "sellers"]:
    print(f"===== raw.{t} =====")
    print(duckdb.sql(f"DESCRIBE raw.{t}"))
    print(duckdb.sql(f"SELECT * FROM raw.{t} LIMIT 5"))

===== raw.customers =====
┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_unique_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_zip_code_prefix │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_city            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_state           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌──────────────────────────────────┬──────────────────────────────────┬──────────────────────────┬───────────────────────┬────────────────┐
│  

In [19]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(DISTINCT customer_unique_id) AS distinct_unique_ids
FROM raw.customers
"""

print(duckdb.sql(query))

┌────────────┬──────────────────────┬─────────────────────┐
│ total_rows │ distinct_customer_id │ distinct_unique_ids │
│   int64    │        int64         │        int64        │
├────────────┼──────────────────────┼─────────────────────┤
│      99441 │                99441 │               96096 │
└────────────┴──────────────────────┴─────────────────────┘



In [ ]:
Customers: Grain and Identifier Check

- Grain: One row represents one customer record created for one order (not one person).
- Primary key: `customer_id` (99441 rows = 99441 distinct IDs)
- `customer_unique_id`: 96096 distinct values, so the same person can appear under several `customer_id` values.
- To count real customers, use `COUNT(DISTINCT customer_unique_id)` = 96096.

In [20]:
query = """
SELECT 'orders.customer_id -> customers' AS relationship, COUNT(*) AS orphans
FROM raw.orders o
ANTI JOIN raw.customers c ON o.customer_id = c.customer_id

UNION ALL
SELECT 'order_items.order_id -> orders', COUNT(*)
FROM raw.order_items oi
ANTI JOIN raw.orders o ON oi.order_id = o.order_id

UNION ALL
SELECT 'order_items.product_id -> products', COUNT(*)
FROM raw.order_items oi
ANTI JOIN raw.products p ON oi.product_id = p.product_id

UNION ALL
SELECT 'order_items.seller_id -> sellers', COUNT(*)
FROM raw.order_items oi
ANTI JOIN raw.sellers s ON oi.seller_id = s.seller_id
"""

print(duckdb.sql(query))

┌────────────────────────────────────┬─────────┐
│            relationship            │ orphans │
│              varchar               │  int64  │
├────────────────────────────────────┼─────────┤
│ orders.customer_id -> customers    │       0 │
│ order_items.order_id -> orders     │       0 │
│ order_items.product_id -> products │       0 │
│ order_items.seller_id -> sellers   │       0 │
└────────────────────────────────────┴─────────┘



In [21]:
query = """
SELECT 'customers -> orders' AS relationship, MIN(n) AS min_children, MAX(n) AS max_children FROM (
    SELECT c.customer_id, COUNT(o.order_id) AS n
    FROM raw.customers c
    LEFT JOIN raw.orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id)

UNION ALL
SELECT 'orders -> order_items', MIN(n), MAX(n) FROM (
    SELECT o.order_id, COUNT(oi.order_item_id) AS n
    FROM raw.orders o
    LEFT JOIN raw.order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id)

UNION ALL
SELECT 'products -> order_items', MIN(n), MAX(n) FROM (
    SELECT p.product_id, COUNT(oi.order_item_id) AS n
    FROM raw.products p
    LEFT JOIN raw.order_items oi ON p.product_id = oi.product_id
    GROUP BY p.product_id)

UNION ALL
SELECT 'sellers -> order_items', MIN(n), MAX(n) FROM (
    SELECT s.seller_id, COUNT(oi.order_item_id) AS n
    FROM raw.sellers s
    LEFT JOIN raw.order_items oi ON s.seller_id = oi.seller_id
    GROUP BY s.seller_id)
"""

print(duckdb.sql(query))

┌─────────────────────────┬──────────────┬──────────────┐
│      relationship       │ min_children │ max_children │
│         varchar         │    int64     │    int64     │
├─────────────────────────┼──────────────┼──────────────┤
│ customers -> orders     │            1 │            1 │
│ orders -> order_items   │            0 │           21 │
│ products -> order_items │            1 │          527 │
│ sellers -> order_items  │            1 │         2033 │
└─────────────────────────┴──────────────┴──────────────┘



In [22]:
query = """
SELECT 'orders.customer_id' AS fk, COUNT(*) - COUNT(customer_id) AS null_fks FROM raw.orders
UNION ALL
SELECT 'order_items.order_id', COUNT(*) - COUNT(order_id) FROM raw.order_items
UNION ALL
SELECT 'order_items.product_id', COUNT(*) - COUNT(product_id) FROM raw.order_items
UNION ALL
SELECT 'order_items.seller_id', COUNT(*) - COUNT(seller_id) FROM raw.order_items
"""

print(duckdb.sql(query))

┌────────────────────────┬──────────┐
│           fk           │ null_fks │
│        varchar         │  int64   │
├────────────────────────┼──────────┤
│ orders.customer_id     │        0 │
│ order_items.order_id   │        0 │
│ order_items.product_id │        0 │
│ order_items.seller_id  │        0 │
└────────────────────────┴──────────┘



In [23]:
query = """
SELECT o.order_status, COUNT(*) AS orders_without_items
FROM raw.orders o
ANTI JOIN raw.order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_status
ORDER BY orders_without_items DESC
"""

print(duckdb.sql(query))


┌──────────────┬──────────────────────┐
│ order_status │ orders_without_items │
│   varchar    │        int64         │
├──────────────┼──────────────────────┤
│ unavailable  │                  603 │
│ canceled     │                  164 │
│ created      │                    5 │
│ invoiced     │                    2 │
│ shipped      │                    1 │
└──────────────┴──────────────────────┘

